# IEEE 57-Bus DC-OPF with Line Switching and Capacity Cuts
**Author:** Jewook Park  
**Date:** October 2025  
**Description:** DC-OPF with line switching and enhanced capacity constraints for IEEE 57-bus system


In [ ]:
import os
os.environ["GRB_LICENSE_FILE"] = "/Users/a/Desktop/VIP/sc-opf/API key/gurobi.lic"
import re, pandas as pd, numpy as np, gurobipy as gp, matplotlib.pyplot as plt, networkx as nx
from gurobipy import GRB
pd.set_option('display.max_rows', 100)
print("Ready!")


Ready!


In [ ]:
def extract_matrix_block(lines, varname):
    in_block, matrix_lines = False, []
    for line in lines:
        if line.strip().startswith(f"{varname} = ["): in_block = True; continue
        if in_block:
            if line.strip().startswith("];"): break
            clean = re.sub(r'%.*', '', line).strip().rstrip(';')
            if clean: matrix_lines.append(clean)
    return np.array([[float(x) for x in line.split()] for line in matrix_lines])

with open('data/pglib_opf_case57_ieee.m', 'r') as f: lines = f.readlines()
bus_df = pd.DataFrame(extract_matrix_block(lines, 'mpc.bus'), columns=['bus_i','type','Pd','Qd','Gs','Bs','area','Vm','Va','baseKV','zone','Vmax','Vmin'])
gen_df = pd.DataFrame(extract_matrix_block(lines, 'mpc.gen'), columns=['bus','Pg','Qg','Qmax','Qmin','Vg','mBase','status','Pmax','Pmin'])
branch_df = pd.DataFrame(extract_matrix_block(lines, 'mpc.branch'), columns=['fbus','tbus','r','x','b','rateA','rateB','rateC','ratio','angle','status','angmin','angmax'])
gencost_df = pd.DataFrame(extract_matrix_block(lines, 'mpc.gencost'), columns=['model','startup','shutdown','n','c2','c1','c0'])
print(f"Loaded: {len(bus_df)} buses, {len(branch_df)} branches")


Loaded: 57 buses, 80 branches


In [ ]:
def solve_dc_opf_capacitycut(bus_df, gen_df, branch_df, gencost_df, capacity_factor=0.9):
    """DC-OPF with capacity cuts (reduced capacity to tighten constraints)"""
    model = gp.Model("DC-OPF-CapacityCut-Case57")
    model.Params.OutputFlag = 0
    ref_bus = bus_df[bus_df['type'] == 3]['bus_i'].values[0]
    M = 1000
    
    Pg = model.addVars(gen_df.index, lb=0, name="Pg")
    theta = model.addVars(bus_df['bus_i'], lb=-GRB.INFINITY, name="theta")
    P_branch = model.addVars(branch_df.index, lb=-GRB.INFINITY, name="P_branch")
    z = model.addVars(branch_df.index, vtype=GRB.BINARY, name="z")
    
    obj = gp.QuadExpr()
    for idx, row in gencost_df.iterrows():
        obj += row['c2'] * Pg[idx]*Pg[idx] + row['c1'] * Pg[idx] + row['c0']
    model.setObjective(obj, GRB.MINIMIZE)
    
    model.addConstr(theta[ref_bus] == 0)
    
    for idx, row in gen_df.iterrows():
        model.addConstr(Pg[idx] >= row['Pmin'])
        model.addConstr(Pg[idx] <= row['Pmax'])
    
    for idx, row in branch_df.iterrows():
        fbus, tbus, x = row['fbus'], row['tbus'], row['x']
        B = 1.0 / x
        model.addConstr(P_branch[idx] - B * (theta[fbus] - theta[tbus]) <= M * (1 - z[idx]))
        model.addConstr(P_branch[idx] - B * (theta[fbus] - theta[tbus]) >= -M * (1 - z[idx]))
        model.addConstr(P_branch[idx] <= M * z[idx])
        model.addConstr(P_branch[idx] >= -M * z[idx])
    
    # Capacity constraints with cut factor
    for idx, row in branch_df.iterrows():
        rateA = row['rateA'] * capacity_factor  # Reduced capacity
        if row['rateA'] > 0:
            model.addConstr(P_branch[idx] <= rateA * z[idx])
            model.addConstr(P_branch[idx] >= -rateA * z[idx])
    
    for idx, row in bus_df.iterrows():
        bus, Pd = row['bus_i'], row['Pd']
        gen_at_bus = gen_df[gen_df['bus'] == bus].index.tolist()
        gen_P = gp.quicksum(Pg[g] for g in gen_at_bus) if gen_at_bus else 0
        branch_out = gp.quicksum(P_branch[br] for br in branch_df[branch_df['fbus'] == bus].index)
        branch_in = gp.quicksum(P_branch[br] for br in branch_df[branch_df['tbus'] == bus].index)
        model.addConstr(gen_P - Pd == branch_out - branch_in)
    
    model.optimize()
    return model, Pg, theta, P_branch, z


In [ ]:
model, Pg, theta, P_branch, z = solve_dc_opf_capacitycut(bus_df, gen_df, branch_df, gencost_df, capacity_factor=0.9)

if model.status == GRB.OPTIMAL:
    print("IEEE 57-BUS DC-OPF WITH CAPACITY CUTS (By Jewook Park)")
    print(f"Cost: ${model.objVal:.2f} | Gen: {sum(Pg[i].X for i in gen_df.index):.2f} MW")
    print(f"Lines ON: {sum(z[i].X > 0.5 for i in branch_df.index)}/{len(branch_df)}")
else:
    print(f"Failed: {model.status}")


Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2725033
Academic license 2725033 - for non-commercial use only - registered to jp___@gatech.edu
IEEE 57-BUS DC-OPF WITH CAPACITY CUTS (By Jewook Park)
Cost: $34772.95 | Gen: 1250.80 MW
Lines ON: 65/80
